<a href="https://www.kaggle.com/code/avikdas567/stanford-rna-3d-folding-part-2?scriptVersionId=294063280" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ============================================================
# Stanford RNA 3D Folding Part 2
# ============================================================

import os
import math
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

# -------------------------
# Config
# -------------------------
DATA_ROOT = "/kaggle/input/stanford-rna-3d-folding-2"
TEST_SEQ_PATH = f"{DATA_ROOT}/test_sequences.csv"
SAMPLE_SUB_PATH = f"{DATA_ROOT}/sample_submission.csv"
OUT_PATH = "submission.csv"

np.random.seed(42)
random.seed(42)

# -------------------------
# Load data
# -------------------------
test_df = pd.read_csv(TEST_SEQ_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

# -------------------------
# Geometry utilities
# -------------------------
def helical_coords(n, radius=8.0, rise=3.4, twist=32.7):
    """
    Generate a simple RNA-like helix backbone (C1' trace).
    Parameters loosely inspired by A-form RNA.
    """
    coords = np.zeros((n, 3), dtype=np.float32)
    for i in range(n):
        angle = np.deg2rad(i * twist)
        x = radius * np.cos(angle)
        y = radius * np.sin(angle)
        z = i * rise
        coords[i] = [x, y, z]
    return coords

def perturb(coords, noise=1.0):
    """Add small Gaussian noise for alternate conformations"""
    return coords + np.random.normal(scale=noise, size=coords.shape)

def center(coords):
    return coords - coords.mean(axis=0, keepdims=True)

# -------------------------
# Build predictions
# -------------------------
rows = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    target_id = row["target_id"]
    seq = row["sequence"]
    L = len(seq)

    # Base helix
    base = center(helical_coords(L))

    # Generate 5 conformations
    preds = []
    for k in range(5):
        conf = perturb(base, noise=0.5 + 0.2 * k)
        conf = np.clip(conf, -999.999, 9999.999)
        preds.append(conf)

    # Write per-residue rows
    for i, nt in enumerate(seq):
        out = {
            "ID": f"{target_id}_{i+1}",
            "resname": nt,
            "resid": i + 1,
        }
        for k in range(5):
            out[f"x_{k+1}"] = preds[k][i, 0]
            out[f"y_{k+1}"] = preds[k][i, 1]
            out[f"z_{k+1}"] = preds[k][i, 2]
        rows.append(out)

# -------------------------
# Create submission
# -------------------------
sub = pd.DataFrame(rows)

# Ensure column order matches sample submission
expected_cols = list(sample_sub.columns)
sub = sub[expected_cols]

sub.to_csv(OUT_PATH, index=False)

print("submission.csv written")
display(sub.head())

100%|██████████| 28/28 [00:00<00:00, 191.04it/s]


submission.csv written


,ID,resname,resid,x_1,y_1,z_1,x_2,y_2,z_2,x_3,y_3,z_3,x_4,y_4,z_4,x_5,y_5,z_5
0,8ZNQ_1,A,1,8.543062,-0.726414,-48.976155,8.362660,0.020770,-49.791436,8.857806,-1.428724,-50.263802,9.880106,-2.236730,-48.020519,8.969856,1.335279,-49.441387
1,8ZNQ_2,C,2,7.788306,3.547564,-46.017066,6.797428,3.390165,-46.924458,7.461016,3.463524,-45.257397,7.038047,2.584981,-45.391684,7.549016,4.561828,-46.421584
2,8ZNQ_3,C,3,4.414558,7.000325,-42.734737,3.832236,6.799346,-42.496421,4.050865,6.551061,-43.262114,3.843917,5.956369,-42.423218,3.916272,6.632977,-42.373021
3,8ZNQ_4,G,4,-0.561225,7.031199,-39.332863,-0.996716,6.272148,-39.394450,-2.195867,6.861044,-38.329240,-1.256350,7.387777,-38.371655,-1.837418,7.294771,-38.452601
4,8ZNQ_5,U,5,-4.811679,4.442039,-36.562456,-5.172560,4.837085,-35.812897,-4.739976,4.277514,-35.544134,-3.188042,4.037082,-33.353660,-3.046173,6.645731,-32.900860
